# HuBMAP metadata for spatial transcriptomics reference scoring

This workflow mirrors `cellxgene_quant.ipynb`, but uses HuBMAP's public APIs. HuBMAP exposes dataset-level metadata through the Search API, not CELLxGENE-style cell-level `obs`, so this notebook normalizes the closest available fields into the same columns used by the existing reference-scoring helpers.

API references:
- Search API parameterized endpoint: https://docs.hubmapconsortium.org/param-search/
- Dataset schema: https://docs.hubmapconsortium.org/param-search/schema-dataset.html
- API overview: https://docs.hubmapconsortium.org/apis.html

## Museum of Spatial Transcriptomics integration

Spatial datasets from the public Museum of Spatial Transcriptomics Google Sheet are normalized into the same dataset-level schema and appended to `spatial_meta` before reference scoring. These rows are spatial-query datasets only; they are not treated as sc/snRNA reference candidates.


In [1]:
import json
import time
import urllib.error
import urllib.parse
import urllib.request
from collections.abc import Iterable

import numpy as np
import pandas as pd

HUBMAP_SEARCH_BASE = "https://search.api.hubmapconsortium.org/v3"

# Dataset types listed in HuBMAP's Dataset schema docs. Querying by type avoids very large
# all-dataset responses and keeps the workflow resilient to 30s API timeouts.
HUBMAP_DATASET_TYPES = [
    "10X Multiome",
    "2D Imaging Mass Cytometry",
    "3D Imaging Mass Cytometry",
    "ATACseq",
    "Auto-fluorescence",
    "CODEX",
    "DESI",
    "GeoMx (NGS)",
    "HiFi-Slide",
    "Histology",
    "LC-MS",
    "Light Sheet",
    "MALDI",
    "MERFISH",
    "MIBI",
    "MUSIC",
    "PhenoCycler",
    "RNAseq",
    "RNAseq (with probes)",
    "SIMS",
    "SNARE-seq2",
    "Second Harmonic Generation (SHG)",
    "seqFISH",
    "Slideseq",
    "Thick section Multiphoton MxIF",
    "Visium (no probes)",
    "Visium (with probes)",
    "WGS",
]

SPATIAL_DATASET_TYPES = {
    "GeoMx (NGS)",
    "HiFi-Slide",
    "MERFISH",
    "seqFISH",
    "Slideseq",
    "Visium (no probes)",
    "Visium (with probes)",
    "Visium",
    "Xenium",
}

SCRNA_REFERENCE_TYPES = {
    "10X Multiome",
    "RNAseq",
    "RNAseq (with probes)",
    "SNARE-seq2",
}

ORGAN_CODE_TO_NAME = {
    "AO": "aorta",
    "BL": "blood",
    "BD": "blood vasculature",
    "BM": "bone marrow",
    "BR": "brain",
    "HT": "heart",
    "LB": "bronchus",
    "LD": "large intestine",
    "LE": "left eye",
    "LF": "left fallopian tube",
    "LI": "liver",
    "LK": "left kidney",
    "LL": "left lung",
    "LN": "lymph node",
    "LO": "left ovary",
    "LY": "lymphatic vasculature",
    "OT": "other",
    "PA": "pancreas",
    "PL": "placenta",
    "RA": "renal artery",
    "RE": "right eye",
    "RF": "right fallopian tube",
    "RK": "right kidney",
    "RL": "right lung",
    "RO": "right ovary",
    "SI": "small intestine",
    "SK": "skin",
    "SP": "spleen",
    "TH": "thymus",
    "TR": "trachea",
    "UR": "ureter",
    "UT": "uterus",
}

ORGAN_NAME_TO_GENERAL = {
    "left kidney": "kidney",
    "right kidney": "kidney",
    "left lung": "lung",
    "right lung": "lung",
    "left ovary": "ovary",
    "right ovary": "ovary",
    "left eye": "eye",
    "right eye": "eye",
    "large intestine": "intestine",
    "small intestine": "intestine",
    "renal artery": "blood vasculature",
}

In [2]:
import importlib
import cellxgene_quantification
importlib.reload(cellxgene_quantification)
from cellxgene_quantification import (
    fetch_museum_spatial_transcriptomics,
    normalize_museum_spatial_transcriptomics,
)

museum_raw = fetch_museum_spatial_transcriptomics()
museum_spatial_meta = normalize_museum_spatial_transcriptomics(museum_raw)
museum_spatial_meta.head()

,dataset_id,source_database,collection_id,collection_name,dataset_title,dataset_type,assays,tissues,tissue_general,diseases,...,pmid,journal,url,accession,data_link,developmental_stage,sex,n_cells_source,dataset_total_cell_count_source,median_n_measured_vars_source
0,museum_st_77dd72643db0,museum_of_spatial_transcriptomics,25417113.0,Genome-wide RNA Tomography in the Zebrafish Em...,Genome-wide RNA Tomography in the Zebrafish Em...,Tomo-seq,"[Tomo-seq, CEL-seq]",[],[],[normal],...,25417113.0,Cell,https://doi.org/10.1016/j.cell.2014.09.038,GSE59873,None,15 somite,None,museum.n_cells/bins/spots,museum.n_cells/bins/spots,None
1,museum_st_d0e3f9f19301,museum_of_spatial_transcriptomics,25417113.0,Genome-wide RNA Tomography in the Zebrafish Em...,Genome-wide RNA Tomography in the Zebrafish Em...,Tomo-seq,"[Tomo-seq, CEL-seq]",[Posterior end],[Posterior end],[normal],...,25417113.0,Cell,https://doi.org/10.1016/j.cell.2014.09.038,GSE59873,None,18 somite,None,None,None,None
2,museum_st_c7052ff7d946,museum_of_spatial_transcriptomics,25417113.0,Genome-wide RNA Tomography in the Zebrafish Em...,Genome-wide RNA Tomography in the Zebrafish Em...,Tomo-seq,"[Tomo-seq, CEL-seq]",[forelimbs],[forelimbs],[normal],...,25417113.0,Cell,https://doi.org/10.1016/j.cell.2014.09.038,GSE59873,None,E10.5,None,None,None,None
3,museum_st_645557f6b6e2,museum_of_spatial_transcriptomics,25417113.0,Genome-wide RNA Tomography in the Zebrafish Em...,Genome-wide RNA Tomography in the Zebrafish Em...,Tomo-seq,"[Tomo-seq, CEL-seq]",[forelimbs],[forelimbs],[normal],...,25417113.0,Cell,https://doi.org/10.1016/j.cell.2014.09.038,GSE59873,None,E10.5,None,None,None,None
4,museum_st_ed65e834ad45,museum_of_spatial_transcriptomics,25417113.0,Genome-wide RNA Tomography in the Zebrafish Em...,Genome-wide RNA Tomography in the Zebrafish Em...,Tomo-seq,"[Tomo-seq, CEL-seq]",[],[],[normal],...,25417113.0,Cell,https://doi.org/10.1016/j.cell.2014.09.038,GSE59873,None,shield stage,None,None,None,None


In [3]:
def fetch_json(path, params=None, timeout=60):
    url = f"{HUBMAP_SEARCH_BASE}{path}"
    if params:
        url = f"{url}?{urllib.parse.urlencode(params)}"

    request = urllib.request.Request(
        url,
        headers={"Accept": "application/json", "User-Agent": "Syndri-HuBMAP-metadata/1.0"},
    )

    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            return json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as exc:
        if exc.code == 404:
            return []
        raise


def fetch_hubmap_datasets(dataset_types=HUBMAP_DATASET_TYPES, status="Published", sleep_s=0.2):
    records = []

    for dataset_type in dataset_types:
        params = {"dataset_type": dataset_type}
        if status is not None:
            params["status"] = status

        print(f"Fetching HuBMAP {dataset_type!r} datasets")
        docs = fetch_json("/param-search/datasets", params=params)
        print(f"  {len(docs):,} records")
        records.extend(docs)
        time.sleep(sleep_s)

    # Deduplicate because derived entities can be reachable through multiple query paths.
    deduped = {}
    for record in records:
        deduped[record.get("uuid") or record.get("hubmap_id")] = record

    return list(deduped.values())

In [4]:
def listify(x):
    if x is None:
        return []
    if isinstance(x, float) and np.isnan(x):
        return []
    if isinstance(x, str):
        return [x] if x.strip() else []
    if isinstance(x, dict):
        return [x]
    if isinstance(x, Iterable):
        return list(x)
    return [x]


def clean_unique(values):
    out = []
    seen = set()
    for value in values:
        if value is None:
            continue
        if isinstance(value, float) and np.isnan(value):
            continue
        text = str(value).strip()
        if not text:
            continue
        key = text.lower()
        if key not in seen:
            out.append(text)
            seen.add(key)
    return sorted(out, key=str.lower)


def find_values_by_key(obj, key_terms):
    values = []
    terms = tuple(term.lower() for term in key_terms)

    if isinstance(obj, dict):
        for key, value in obj.items():
            key_text = str(key).lower()
            if any(term in key_text for term in terms):
                if isinstance(value, (dict, list)):
                    values.extend(find_leaf_values(value))
                else:
                    values.append(value)
            values.extend(find_values_by_key(value, terms))
    elif isinstance(obj, list):
        for item in obj:
            values.extend(find_values_by_key(item, terms))

    return values


def find_leaf_values(obj):
    if isinstance(obj, dict):
        values = []
        for value in obj.values():
            values.extend(find_leaf_values(value))
        return values
    if isinstance(obj, list):
        values = []
        for item in obj:
            values.extend(find_leaf_values(item))
        return values
    return [obj]


def parse_numeric(value):
    try:
        if value in [None, ""]:
            return np.nan
        return float(str(value).replace(",", ""))
    except (TypeError, ValueError):
        return np.nan


def timestamp_ms_to_datetime(value):
    if value in [None, ""]:
        return pd.NaT
    return pd.to_datetime(value, unit="ms", errors="coerce")


def extract_organs(record):
    codes = []
    names = []

    for sample in listify(record.get("origin_samples")) + listify(record.get("origin_sample")):
        if not isinstance(sample, dict):
            continue
        code = sample.get("organ")
        if code:
            codes.append(code)
            names.append(ORGAN_CODE_TO_NAME.get(str(code).upper(), code))
        names.extend(find_values_by_key(sample.get("metadata", {}), ["organ", "tissue", "anatomy"]))

    return clean_unique(names), clean_unique(codes)


def extract_diseases(record):
    values = []
    values.extend(find_values_by_key(record.get("metadata", {}), ["disease", "diagnosis", "condition"]))
    values.extend(find_values_by_key(record.get("donor", {}).get("metadata", {}), ["disease", "diagnosis", "condition", "health"]))
    for sample in listify(record.get("origin_samples")) + listify(record.get("source_samples")):
        if isinstance(sample, dict):
            values.extend(find_values_by_key(sample.get("metadata", {}), ["disease", "diagnosis", "condition", "health"]))

    diseases = clean_unique(values)
    return diseases if diseases else ["normal"]


def extract_cell_types(record):
    metadata = record.get("metadata", {}) or {}
    values = []
    sources = []

    # HuBMAP targeted spatial assays can identify a targeted biological entity.
    # Treat it as a cell type only when the paired ontology ID is a Cell Ontology term.
    target_ids = listify(metadata.get("targeted_entity_id"))
    target_labels = listify(metadata.get("targeted_entity_label"))
    for target_id, target_label in zip(target_ids, target_labels):
        if str(target_id).upper().startswith("CL:") and str(target_label).strip():
            values.append(target_label)
            sources.append("metadata.targeted_entity_label")

    # Common explicit annotation fields. Avoid broad anatomy/CCF annotations here;
    # those are usually tissue structures, not cell types.
    explicit_keys = [
        "cell_type",
        "cell_type_label",
        "cell_type_name",
        "cell_types",
        "cell_types_label",
        "celltypes",
        "asct_b_cell_type",
    ]

    for key in explicit_keys:
        if key in metadata:
            extracted = clean_unique(find_leaf_values(metadata[key]))
            values.extend(extracted)
            sources.extend([f"metadata.{key}"] * len(extracted))

    ingest_metadata = record.get("ingest_metadata", {}) or {}
    for key in explicit_keys:
        if key in ingest_metadata:
            extracted = clean_unique(find_leaf_values(ingest_metadata[key]))
            values.extend(extracted)
            sources.extend([f"ingest_metadata.{key}"] * len(extracted))

    values = clean_unique(values)
    return values, clean_unique(sources)


CELL_COUNT_KEYS = [
    # Best available HuBMAP RNA-seq count fields observed in metadata.
    "number_of_input_cells_or_nuclei",
    "sc_isolation_cell_number",
    # Spatial transcriptomics often reports spots rather than cells.
    "number_of_spots",
    # Capture counts are useful but can be expected/nominal values.
    "expected_entity_capture_count",
    "number_of_cells",
    "cell_count",
    "n_cells",
]

MEASURED_VAR_KEYS = [
    # HuBMAP does not consistently expose gene/feature counts for RNA-seq through
    # Search API metadata. These keys catch the assay types that do publish a
    # directly comparable feature count, especially multiplexed imaging.
    "number_of_genes",
    "number_of_features",
    "number_of_probes",
    "number_of_targets",
    "number_of_antibodies",
    "number_of_biomarkers",
    "n_genes",
    "n_features",
]


def extract_first_numeric_field(record, candidate_keys):
    metadata = record.get("metadata", {}) or {}

    for key in candidate_keys:
        value = parse_numeric(metadata.get(key))
        if not pd.isna(value):
            return value, f"metadata.{key}"

    return np.nan, None


def extract_n_cells(record):
    return extract_first_numeric_field(record, CELL_COUNT_KEYS)


def extract_n_measured_vars(record):
    return extract_first_numeric_field(record, MEASURED_VAR_KEYS)


def infer_suspension_type(record):
    metadata = record.get("metadata", {}) or {}
    text = " ".join(str(metadata.get(key, "")) for key in ["assay_input_entity", "analyte_class", "preparation_protocol_doi"])
    text = text.lower()
    if "nuclei" in text or "nucleus" in text:
        return ["nucleus"]
    if "cell" in text:
        return ["cell"]
    if record.get("dataset_type") in SCRNA_REFERENCE_TYPES:
        return ["cell_or_nucleus"]
    return []


def normalize_hubmap_dataset(record):
    tissues, organ_codes = extract_organs(record)
    tissue_general = clean_unique(ORGAN_NAME_TO_GENERAL.get(t.lower(), t) for t in tissues)
    cell_types, cell_types_source = extract_cell_types(record)
    n_cells, n_cells_source = extract_n_cells(record)
    n_measured_vars, n_measured_vars_source = extract_n_measured_vars(record)
    dataset_type = record.get("dataset_type") or record.get("display_subtype")
    donor = record.get("donor", {}) or {}

    return {
        "dataset_id": record.get("uuid"),
        "hubmap_id": record.get("hubmap_id"),
        "collection_id": record.get("group_uuid"),
        "collection_name": record.get("group_name"),
        "dataset_title": record.get("title"),
        "assays": clean_unique([dataset_type]),
        "dataset_type": dataset_type,
        "tissues": tissues,
        "tissue_general": tissue_general,
        "organ_codes": organ_codes,
        "diseases": extract_diseases(record),
        "cell_types": cell_types,
        "cell_types_source": cell_types_source,
        "organism": "Homo sapiens",
        "organisms": ["Homo sapiens"],
        "suspension_types": infer_suspension_type(record),
        "donor_id": donor.get("uuid") or donor.get("hubmap_id"),
        "n_donors": int(bool(donor.get("uuid") or donor.get("hubmap_id"))),
        "n_cells": n_cells,
        "n_cells_source": n_cells_source,
        "dataset_total_cell_count": n_cells,
        "dataset_total_cell_count_source": n_cells_source,
        "median_n_measured_vars": n_measured_vars,
        "median_n_measured_vars_source": n_measured_vars_source,
        "status": record.get("status"),
        "data_access_level": record.get("data_access_level"),
        "registered_doi": record.get("registered_doi"),
        "doi_url": record.get("doi_url"),
        "created_at": timestamp_ms_to_datetime(record.get("created_timestamp")),
        "published_at": timestamp_ms_to_datetime(record.get("published_timestamp")),
        "metadata": record.get("metadata", {}),
        "raw_record": record,
    }

## Cell-type metadata caveat

HuBMAP Search API dataset records do not consistently expose CELLxGENE-style per-dataset cell-type annotation sets. This notebook parses explicit cell-type metadata when present and accepts `targeted_entity_label` only when the paired `targeted_entity_id` is a Cell Ontology `CL:` term. Broad CCF/RUI and anatomical structure annotations are intentionally excluded from `cell_types` because they usually describe tissue regions, not cell types.

In [5]:
hubmap_records = fetch_hubmap_datasets()
len(hubmap_records)

Fetching HuBMAP '10X Multiome' datasets
  155 records
Fetching HuBMAP '2D Imaging Mass Cytometry' datasets
  13 records
Fetching HuBMAP '3D Imaging Mass Cytometry' datasets
  3 records
Fetching HuBMAP 'ATACseq' datasets
  500 records
Fetching HuBMAP 'Auto-fluorescence' datasets
  356 records
Fetching HuBMAP 'CODEX' datasets
  128 records
Fetching HuBMAP 'DESI' datasets
  15 records
Fetching HuBMAP 'GeoMx (NGS)' datasets
  500 records
Fetching HuBMAP 'HiFi-Slide' datasets
  0 records
Fetching HuBMAP 'Histology' datasets
  500 records
Fetching HuBMAP 'LC-MS' datasets
  474 records
Fetching HuBMAP 'Light Sheet' datasets
  11 records
Fetching HuBMAP 'MALDI' datasets
  151 records
Fetching HuBMAP 'MERFISH' datasets
  0 records
Fetching HuBMAP 'MIBI' datasets
  257 records
Fetching HuBMAP 'MUSIC' datasets
  14 records
Fetching HuBMAP 'PhenoCycler' datasets
  142 records
Fetching HuBMAP 'RNAseq' datasets
  500 records
Fetching HuBMAP 'RNAseq (with probes)' datasets
  500 records
Fetching HuBM

4704

In [7]:
summary = pd.DataFrame([normalize_hubmap_dataset(record) for record in hubmap_records])

# Match the CELLxGENE workflow convention: rows are dataset-level summaries with these
# standard columns available to the shared scoring functions.
summary = summary.sort_values(["dataset_type", "dataset_title"], na_position="last").reset_index(drop=True)
summary["source_database"] = "hubmap"
summary.to_csv('hubmap_meta.csv')

In [ ]:
def preview_list(x, n=4):
    if not isinstance(x, list):
        return x
    if len(x) <= n:
        return ", ".join(map(str, x))
    return ", ".join(map(str, x[:n])) + f", ... ({len(x)} total)"


display_cols = [
    "dataset_id",
    "hubmap_id",
    "collection_name",
    "dataset_title",
    "dataset_type",
    "n_cells",
    "n_cells_source",
    "median_n_measured_vars",
    "median_n_measured_vars_source",
    "cell_types",
    "cell_types_source",
    "assays",
    "tissues",
    "tissue_general",
    "diseases",
    "suspension_types",
    "n_donors",
    "data_access_level",
]

summary_display = summary.copy()
for col in ["assays", "tissues", "tissue_general", "diseases", "suspension_types", "cell_types", "cell_types_source"]:
    summary_display[col] = summary_display[col].apply(preview_list)

summary_display[display_cols].sort_values("n_cells", ascending=False, na_position="last").head(20)

In [ ]:
count_coverage = pd.DataFrame({
    "non_null_fraction": summary[
        ["dataset_total_cell_count", "median_n_measured_vars"]
    ].notna().mean(),
    "non_null_count": summary[
        ["dataset_total_cell_count", "median_n_measured_vars"]
    ].notna().sum(),
    "total_datasets": len(summary),
})

count_sources = {
    "dataset_total_cell_count_source": summary["dataset_total_cell_count_source"].value_counts(dropna=False),
    "median_n_measured_vars_source": summary["median_n_measured_vars_source"].value_counts(dropna=False),
}

display(count_coverage)
display(count_sources["dataset_total_cell_count_source"].head(20))
display(count_sources["median_n_measured_vars_source"].head(20))

In [ ]:
cell_type_coverage = pd.DataFrame({
    "non_empty_fraction": [summary["cell_types"].apply(bool).mean()],
    "non_empty_count": [summary["cell_types"].apply(bool).sum()],
    "total_datasets": [len(summary)],
})

cell_type_sources = (
    summary[["dataset_id", "cell_types_source"]]
    .assign(cell_types_source=lambda df: df["cell_types_source"].apply(lambda x: x if isinstance(x, list) else []))
    .explode("cell_types_source")
    .dropna()
    .groupby("cell_types_source")
    .size()
    .sort_values(ascending=False)
)

display(cell_type_coverage)
display(cell_type_sources.head(20))
summary_display[summary["cell_types"].apply(bool)][[
    "dataset_id",
    "hubmap_id",
    "dataset_type",
    "dataset_title",
    "cell_types",
    "cell_types_source",
    "tissues",
]].head(30)

In [ ]:
import matplotlib.pyplot as plt

def explode_list_column(df, col):
    tmp = df[["dataset_id", col]].copy()
    tmp[col] = tmp[col].apply(lambda x: x if isinstance(x, list) else [])
    return tmp.explode(col).dropna()


tissue_counts = (
    explode_list_column(summary, "tissue_general")
    .query("tissue_general != ''")
    .groupby("tissue_general")
    .size()
    .sort_values(ascending=False)
    .head(20)
    .sort_values()
)

plt.figure(figsize=(8, 6))
plt.barh(tissue_counts.index, tissue_counts.values)
plt.xlabel("Number of datasets")
plt.ylabel("General tissue")
plt.title("HuBMAP: Top represented tissue categories")
plt.tight_layout()
plt.show()

type_counts = summary["dataset_type"].value_counts().sort_values().tail(20)

plt.figure(figsize=(8, 6))
plt.barh(type_counts.index, type_counts.values)
plt.xlabel("Number of datasets")
plt.ylabel("Dataset type")
plt.title("HuBMAP: Top dataset types")
plt.tight_layout()
plt.show()

In [ ]:
import importlib
import cellxgene_quantification
importlib.reload(cellxgene_quantification)
from cellxgene_quantification import find_best_references_for_spatial_dataset

hubmap_spatial_meta = summary[summary["dataset_type"].isin(SPATIAL_DATASET_TYPES)].copy()
spatial_meta = pd.concat(
    [hubmap_spatial_meta, museum_spatial_meta],
    ignore_index=True,
    sort=False,
)
ref_meta = summary[
    summary["dataset_type"].isin(SCRNA_REFERENCE_TYPES)
    & summary["n_cells"].fillna(0).gt(0)
].copy()

spatial_meta[
    [
        "source_database",
        "dataset_id",
        "hubmap_id",
        "dataset_title",
        "collection_name",
        "assays",
        "tissues",
        "tissue_general",
        "diseases",
        "dataset_total_cell_count",
        "n_cells",
    ]
]

In [ ]:
all_scores = []

for _, st_row in spatial_meta.iterrows():
    top_refs = find_best_references_for_spatial_dataset(
        st_row=st_row,
        ref_meta=ref_meta,
        top_k=10,
    )
    top_refs["spatial_hubmap_id"] = st_row.get("hubmap_id")
    top_refs["spatial_dataset_title"] = st_row.get("dataset_title")
    all_scores.append(top_refs)

reference_scores = pd.concat(all_scores, ignore_index=True) if all_scores else pd.DataFrame()
reference_scores

In [ ]:
import matplotlib.pyplot as plt

def plot_reference_score_report(reference_scores):
    if reference_scores.empty:
        return reference_scores

    scores = reference_scores.copy()
    numeric_cols = [
        "score",
        "tissue_score",
        "disease_score",
        "celltype_score",
        "quality_score",
        "same_study_score",
        "ref_n_cells",
        "ref_n_donors",
    ]

    for col in numeric_cols:
        if col in scores.columns:
            scores[col] = pd.to_numeric(scores[col], errors="coerce")

    best_refs = (
        scores
        .sort_values(["spatial_dataset_id", "score"], ascending=[True, False])
        .groupby("spatial_dataset_id")
        .head(1)
        .reset_index(drop=True)
    )

    component_cols = [
        "tissue_score",
        "disease_score",
        "celltype_score",
        "quality_score",
        "same_study_score",
    ]

    plt.figure(figsize=(7, 4))
    plt.hist(best_refs["score"].dropna(), bins=25)
    plt.xlabel("Best available scRNA-seq reference score")
    plt.ylabel("Number of spatial datasets")
    plt.title("HuBMAP: Distribution of best reference scores")
    plt.xlim(0, 1)
    plt.tight_layout()
    plt.show()

    if "tier" in best_refs.columns:
        tier_counts = best_refs["tier"].value_counts()
        plt.figure(figsize=(8, 4))
        plt.barh(tier_counts.index[::-1], tier_counts.values[::-1])
        plt.xlabel("Number of spatial datasets")
        plt.ylabel("Reference tier")
        plt.title("HuBMAP: Best reference tier per spatial dataset")
        plt.tight_layout()
        plt.show()

    existing_components = [c for c in component_cols if c in best_refs.columns]
    component_means = best_refs[existing_components].mean().sort_values()

    plt.figure(figsize=(7, 4))
    plt.barh(component_means.index, component_means.values)
    plt.xlabel("Mean component score")
    plt.ylabel("Score component")
    plt.title("HuBMAP: Average score components among best references")
    plt.xlim(0, 1)
    plt.tight_layout()
    plt.show()

    return best_refs


best_refs = plot_reference_score_report(reference_scores)
best_refs

In [ ]:
if not reference_scores.empty:
    display(reference_scores[
        ["score", "tissue_score", "disease_score", "celltype_score", "quality_score", "same_study_score"]
    ].describe())

    display(pd.Series({
        "frac_score_ge_0.8": (reference_scores["score"] >= 0.8).mean(),
        "frac_tissue_1": (reference_scores["tissue_score"] == 1).mean(),
        "frac_disease_1": (reference_scores["disease_score"] == 1).mean(),
        "frac_celltype_ge_0.9": (reference_scores["celltype_score"] >= 0.9).mean(),
        "frac_same_study_1": (reference_scores["same_study_score"] == 1).mean(),
    }))

In [ ]:
if not reference_scores.empty:
    reference_scores.to_csv("hubmap_reference_scores.csv", index=False)
summary.drop(columns=["raw_record"]).to_csv("hubmap_metadata_summary.csv", index=False)
museum_spatial_meta.to_csv("museum_spatial_metadata_summary.csv", index=False)
spatial_meta.to_csv("hubmap_plus_museum_spatial_metadata.csv", index=False)

summary.shape, museum_spatial_meta.shape, spatial_meta.shape, reference_scores.shape